## Installing

In [ ]:
!git clone https://github.com/DependableSystemsLab/SolidiFI-benchmark.git
!git clone https://github.com/smartbugs/smartbugs-curated.git
!git clone https://github.com/wuhongjun15/Peculiar.git
!git clone https://github.com/MRdoulestar/DeeSCVHunter.git
!git clone https://github.com/smartbugs/smartbugs-wild.git

Cloning into 'SolidiFI-benchmark'...
remote: Enumerating objects: 2690, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 2690 (delta 27), reused 81 (delta 26), pack-reused 2607 (from 1)
Receiving objects: 100% (2690/2690), 7.29 MiB | 5.69 MiB/s, done.
Resolving deltas: 100% (2134/2134), done.
Updating files: 100% (5107/5107), done.
Cloning into 'smartbugs-curated'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 225 (delta 29), reused 19 (delta 19), pack-reused 177 (from 1)
Receiving objects: 100% (225/225), 173.89 KiB | 908.00 KiB/s, done.
Resolving deltas: 100% (65/65), done.
Cloning into 'Peculiar'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 85 (delta 2), reused 11 (delta 2), pack-reused 74 (from 1)
Receiving obje

In [ ]:
!git clone https://github.com/MANDO-Project/ge-sc

Cloning into 'ge-sc'...
remote: Enumerating objects: 18779, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18779 (delta 5), reused 5 (delta 1), pack-reused 18763 (from 1)
Receiving objects: 100% (18779/18779), 2.29 GiB | 23.68 MiB/s, done.
Resolving deltas: 100% (11208/11208), done.
Updating files: 100% (26632/26632), done.


In [ ]:
import re
import random
import os
random.seed(42)

# Keywords of Solidity; immutable set
keywords = frozenset({
    'bool', 'break', 'case', 'catch', 'const', 'continue', 'default', 'do', 'double', 'struct',
    'else', 'enum', 'payable', 'function', 'modifier', 'emit', 'export', 'extern', 'false', 'constructor',
    'float', 'if', 'contract', 'int', 'long', 'string', 'super', 'or', 'private', 'protected', 'noReentrancy',
    'public', 'return', 'returns', 'assert', 'event', 'indexed', 'using', 'require', 'uint', 'onlyDaoChallenge',
    'transfer', 'Transfer', 'Transaction', 'switch', 'pure', 'view', 'this', 'throw', 'true', 'try', 'revert',
    'bytes', 'bytes4', 'bytes32', 'internal', 'external', 'union', 'constant', 'while', 'for', 'notExecuted',
    'NULL', 'uint256', 'uint128', 'uint8', 'uint16', 'address', 'call', 'msg', 'value', 'sender', 'notConfirmed',
    'private', 'onlyOwner', 'internal', 'onlyGovernor', 'onlyCommittee', 'onlyAdmin', 'onlyPlayers', 'ownerExists',
    'onlyManager', 'onlyHuman', 'only_owner', 'onlyCongressMembers', 'preventReentry', 'noEther', 'onlyMembers',
    'onlyProxyOwner', 'confirmed', 'mapping', 'solidity'
})

global_vars = frozenset({
    'block.timestamp', 'now', 'msg.sender', 'msg.value', 'block.number', 'block.difficulty',
    'block.coinbase', 'block.gaslimit', 'tx.origin', 'tx.gasprice', 'gasleft', 'this', 'super'
})

# Known non-user-defined functions; immutable set
main_set = frozenset({'function', 'constructor', 'modifier', 'contract'})
main_args = frozenset({'argc', 'argv'})

def clean_fragment(fragment):
    fun_symbols = {}
    var_symbols = {}
    fun_count = 1
    var_count = 1

    rx_fun = re.compile(r'\b([_A-Za-z]\w*)\b(?=\s*\()')
    rx_var = re.compile(r'\b([_A-Za-z]\w*)\b(?:(?=\s*\w+\()|(?!\s*\w+))(?!\s*\()')

    cleaned_fragment = []

    for line in fragment:
        # Skip lines that are comments (single-line or multi-line)
        if line.strip().startswith('//') or line.strip().startswith('/*') or line.strip().endswith('*/'):
            cleaned_fragment.append(line)
            continue

        # Remove string literals and non-ASCII chars from the code part
        nostrlit_line = re.sub(r'".*?"', '""', line)
        nocharlit_line = re.sub(r"'.*?'", "''", nostrlit_line)
        ascii_line = re.sub(r'[^\x00-\x7f]', '', nocharlit_line)

        # Process function and variable names
        user_fun = rx_fun.findall(ascii_line)
        user_var = rx_var.findall(ascii_line)

        for fun_name in user_fun:
            if fun_name not in main_set and fun_name not in keywords:
                if fun_name not in fun_symbols:
                    fun_symbols[fun_name] = f'FUN{fun_count}'
                    fun_count += 1
                ascii_line = re.sub(r'\b' + fun_name + r'\b(?=\s*\()', fun_symbols[fun_name], ascii_line)

        for var_name in user_var:
            if var_name not in keywords and var_name not in main_args and var_name not in global_vars:
                if var_name not in var_symbols:
                    var_symbols[var_name] = f'VAR{var_count}'
                    var_count += 1
                ascii_line = re.sub(r'\b' + var_name + r'\b(?:(?=\s*\w+\()|(?!\s*\w+))(?!\s*\()', var_symbols[var_name], ascii_line)

        cleaned_fragment.append(ascii_line)

    return cleaned_fragment

def remove_comments(solidity_code):
    # Regex patterns to match single-line and multi-line comments
    single_line_comment_pattern = r"//.*?(?=\n|$)"
    multi_line_comment_pattern = r"/\*.*?\*/"

    # Remove single-line comments
    code_without_single_line_comments = re.sub(single_line_comment_pattern, '', solidity_code, flags=re.DOTALL)

    # Remove multi-line comments
    cleaned_code = re.sub(multi_line_comment_pattern, '', code_without_single_line_comments, flags=re.DOTALL)

    return cleaned_code.strip()

def clean_code_formatting(code):
    # Split code into lines
    lines = code.split('\n')

    cleaned_lines = []
    for line in lines:
        # Remove trailing whitespace
        line = line.rstrip()

        # Skip empty lines (preserve newlines with content)
        if not line.strip():
            continue

        # Collapse multiple spaces/tabs (except in string literals)
        line = re.sub(r'(?<!")\s{2,}(?!")', ' ', line)

        cleaned_lines.append(line)

    # Join lines while preserving single newlines
    return '\n'.join(cleaned_lines)

In [ ]:
def seed_everything(seed: int):
    # Hàm đặt seed cố định cho tất cả các thư viện để kết quả có thể tái tạo lại được
    import random, os
    import numpy as np
    import torch

    random.seed(seed)  # Cố định seed cho thư viện random
    # os.environ['PYTHONHASHSEED'] = str(seed)  # Cố định hash seed của Python
    np.random.seed(seed)  # Cố định seed cho NumPy
    torch.manual_seed(seed)  # Cố định seed cho PyTorch CPU
    torch.cuda.manual_seed(seed)  # Cố định seed cho PyTorch GPU
    torch.backends.cudnn.deterministic = True  # Đảm bảo tính nhất quán của thuật toán cuDNN
    torch.backends.cudnn.benchmark = True  # Tối ưu hóa tốc độ tính toán

In [ ]:
seed_everything(42)

## SolidiFI-Benchmark

In [ ]:
import os

root = "/content/SolidiFI-benchmark/buggy_contracts"
vulnerabilities = os.listdir(root)
solidiFI_files =  {}
for vulnerability in vulnerabilities:
  solidiFI_files[vulnerability] = {}

for vulnerability in vulnerabilities:
    for file in os.listdir(f"{root}/{vulnerability}"):
        if file.endswith(".sol"):
            sol_code = open(f"{root}/{vulnerability}/{file}", "r").readlines()
            sol_code = "\n".join(sol_code)
            file_name = file.split(".")[0]
            solidiFI_files[vulnerability][file_name] = sol_code

In [ ]:
for vulnerability in vulnerabilities:
    tmp = [solidiFI_files[vulnerability][i] for i in solidiFI_files[vulnerability].keys()]
    print(f"Number of source code with {vulnerability}", len(set(tmp)))

Number of source code with TOD 49
Number of source code with Overflow-Underflow 49
Number of source code with Unchecked-Send 49
Number of source code with Re-entrancy 49
Number of source code with Unhandled-Exceptions 49
Number of source code with tx.origin 49
Number of source code with Timestamp-Dependency 49


In [ ]:
print(clean_code_formatting(remove_comments(solidiFI_files["Timestamp-Dependency"]["buggy_21"])))

pragma solidity ^0.5.11;
contract Token {
 function transfer(address to, uint256 value) public returns (bool success);
address winner_tmstmp7;
function play_tmstmp7(uint startTime) public {
	uint _vtime = block.timestamp;
	if (startTime + (5 * 1 days) == _vtime){
 winner_tmstmp7 = msg.sender;}}
 function transferFrom(address from, address to, uint256 value) public returns (bool success);
address winner_tmstmp23;
function play_tmstmp23(uint startTime) public {
	uint _vtime = block.timestamp;
	if (startTime + (5 * 1 days) == _vtime){
 winner_tmstmp23 = msg.sender;}}
 function balanceOf(address account) external view returns(uint256);
address winner_tmstmp14;
function play_tmstmp14(uint startTime) public {
	if (startTime + (5 * 1 days) == block.timestamp){
 winner_tmstmp14 = msg.sender;}}
 function allowance(address _owner, address _spender)external view returns(uint256);
address winner_tmstmp30;
function play_tmstmp30(uint startTime) public {
	if (startTime + (5 * 1 days) == block.timest

In [ ]:
import re
import pandas as pd

def find_function(lines, error_line):
    function_pattern = re.compile(r'\bfunction\s+\w+\s*\([^)]*\)\s*(\{[^}]*\})?')
    modifier_pattern = re.compile(r'\bmodifier\s+\w+\s*\([^)]*\)\s*(\{[^}]*\})?')
    constructor_pattern = re.compile(r'\bconstructor\s*\([^)]*\)\s*(\{[^}]*\})?')
    start_line = None

    # Search backward for the start of the function
    for i in range(error_line - 1, -1, -1):
        if function_pattern.search(lines[i]) or modifier_pattern.search(lines[i]) or constructor_pattern.search(lines[i]):
            start_line = i
            break

    if start_line is None:
        raise Exception("Function definition not found.")

    # Search forward for the end of the function
    end_line = None
    brace_count = 0
    in_function = False

    for i in range(start_line, len(lines)):
        line = lines[i]
        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_function = True
        elif brace_count == 0 and in_function:
            end_line = i
            break

    if end_line is None:
        raise Exception("Function end not found.")

    # Extract the function code
    function_code = "".join(lines[start_line:end_line + 1])
    return function_code


solidiFI_funcs =  {}
for vulnerability in vulnerabilities:
    solidiFI_funcs[vulnerability] = {
        "vulnerable": [],
        "non-vulnerable": []
    }


for vulnerability in vulnerabilities:
    for name, file in solidiFI_files[vulnerability].items():
        lines = file.splitlines(keepends=True)
        file_num = name[6:]
        csv_path = f"/content/SolidiFI-benchmark/buggy_contracts/{vulnerability}/BugLog_{file_num}.csv"
        tdf = pd.read_csv(csv_path)
        vul_line_num = list(tdf['loc'])

        vul_funcs = []
        all_funcs = []

        for line in range(len(lines)):
            try:
                func = find_function(lines, line)
                if "contract" not in func:
                    func = remove_comments(func).replace("\n\n\n\n\n", "\n").replace("\n\n\n\n", "\n").replace("\n\n\n", "\n").replace("\n\n", "\n")
                    all_funcs.append(func.strip())
            except:
                pass
        for er_line in vul_line_num:
            try:
                func = find_function(lines, er_line)
                if file_num == "21" and vulnerability == "Timestamp-Dependency":
                    print(func)
                    print("-" * 100)
                if "contract" not in func:
                    func = remove_comments(func).replace("\n\n\n\n\n", "\n").replace("\n\n\n\n", "\n").replace("\n\n\n", "\n").replace("\n\n", "\n")
                    vul_funcs.append(func.strip())
            except:
                pass

        vul_funcs = list(set(vul_funcs))
        all_funcs = list(set(all_funcs) - set(vul_funcs))
        solidiFI_funcs[vulnerability]["vulnerable"].extend(vul_funcs)
        solidiFI_funcs[vulnerability]["non-vulnerable"].extend(all_funcs)

      function mul(uint256 a, uint256 b) internal pure returns (uint256) 

    {

        if (a == 0) {

        return 0;}

        uint256 c = a * b;

        assert(c / a == b);

        return c;

    }

----------------------------------------------------------------------------------------------------
  function bug_tmstmp25() view public returns (bool) {

    return block.timestamp >= 1546300800;

  }

----------------------------------------------------------------------------------------------------
function bug_tmstmp40 () public payable {

	uint pastBlockTime_tmstmp40; // Forces one bet per block

	require(msg.value == 10 ether); // must send 10 ether to play

        require(now != pastBlockTime_tmstmp40); // only 1 transaction per block   //bug

        pastBlockTime_tmstmp40 = now;       //bug

        if(now % 15 == 0) { // winner    //bug

            msg.sender.transfer(address(this).balance);

        }

    }

---------------------------------------------------------

In [ ]:
for vulnerability in vulnerabilities:
  print(f"Number of vulnerable functions with {vulnerability}", len(solidiFI_funcs[vulnerability]["vulnerable"]))
  print(f"Number of non-vulnerable functions with {vulnerability}", len(solidiFI_funcs[vulnerability]["non-vulnerable"]))
  print("-" * 100)

Number of vulnerable functions with TOD 1163
Number of non-vulnerable functions with TOD 2462
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Overflow-Underflow 874
Number of non-vulnerable functions with Overflow-Underflow 1716
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Unchecked-Send 676
Number of non-vulnerable functions with Unchecked-Send 1542
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Re-entrancy 773
Number of non-vulnerable functions with Re-entrancy 1297
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Unhandled-Exceptions 814
Number of non-vulnerable functions with Unhandled-Exceptions 1483
----------------------------

In [ ]:
print(clean_code_formatting(solidiFI_funcs["Timestamp-Dependency"]["vulnerable"][20]))
cnt = 0
for function in solidiFI_funcs["Timestamp-Dependency"]["vulnerable"]:
    if "block.timestamp" in function:
        cnt += 1

print(cnt)

function approve(address spender, uint tokens) public returns(bool){
 require(balances[msg.sender] >= tokens);
 require(tokens > 0);
 allowed[msg.sender][spender] = tokens;
 emit Approval(msg.sender, spender, tokens);
 return true;
 }
359


In [ ]:
cnt = 0
for function in solidiFI_funcs["Re-entrancy"]["vulnerable"]:
    if "call.value" in function:
        cnt += 1

print(cnt)

124


## SmartBugs_Curated

In [ ]:
import json

with open("/content/smartbugs-curated/vulnerabilities.json") as f:
    data = json.load(f)

smartbugs_files, smartbugs_functions = {}, {}

for file_dis in data:
    vulnerability = file_dis["path"].split("/")[1]
    file_name = file_dis["path"].split("/")[-1]
    file_name = file_name.split(".")[0]
    if vulnerability not in smartbugs_files:
        smartbugs_files[vulnerability] = {}
        smartbugs_functions[vulnerability] = {
            "vulnerable": [],
            "non-vulnerable": []
        }
    with open(f"/content/smartbugs-curated/{file_dis['path']}", "r") as f:
        code = f.read()
    ok = 0
    lines = code.splitlines(keepends=True)
    code = "\n".join(lines)
    code = remove_comments(code)
    for er_line in file_dis["vulnerabilities"]:
        try:
            func = find_function(lines, er_line["lines"][0])
            if vulnerability == "reentrancy":
                if "call.value" in func:
                    ok = 1
            elif vulnerability == "time_manipulation":
                if "now" in func or "block.timestamp" in func:
                    ok = 1
            if ok == 1:
                smartbugs_functions[vulnerability]["vulnerable"].append(remove_comments(func.strip()))
        except Exception as e:
            print(e)
            pass
    if ok == 1:
        smartbugs_files[vulnerability][code] = file_name

Function definition not found.
Function definition not found.


In [ ]:
for vulnerability in smartbugs_files.keys():
    smartbugs_functions[vulnerability]["vulnerable"] = list(set(smartbugs_functions[vulnerability]["vulnerable"]))
    if vulnerability == "reentrancy" or vulnerability == "time_manipulation":
        print(f"Number of source code with {vulnerability}", len(smartbugs_files[vulnerability]))
        print(f"Number of vulnerable functions with {vulnerability}", len(smartbugs_functions[vulnerability]["vulnerable"]))
        print("-" * 100)

Number of source code with reentrancy 28
Number of vulnerable functions with reentrancy 14
----------------------------------------------------------------------------------------------------
Number of source code with time_manipulation 5
Number of vulnerable functions with time_manipulation 6
----------------------------------------------------------------------------------------------------


## Peculiar Dataset

In [ ]:
!unzip /content/Peculiar/dataset.zip -d /content/

Archive:  /content/Peculiar/dataset.zip
  inflating: /content/data.jsonl     
  inflating: /content/test.txt       
  inflating: /content/train.txt      
  inflating: /content/valid.txt      


In [ ]:
import json

def read_jsonl(file_path):
  data = []
  try:
    with open(file_path, 'r') as f:
      for line in f:
        try:
          data.append(json.loads(line))
        except json.JSONDecodeError as e:
          print(f"Skipping invalid JSON line: {line.strip()}. Error: {e}")
    return data
  except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    return None

# Example usage:
file_path = '/content/data.jsonl' # replace with actual path
data = read_jsonl(file_path)

In [ ]:
def merge_contracts(contracts):
    if not contracts:
        return ""

    pragma_set = set()
    extracted_contracts = []

    # Helper functions to extract contracts and their names
    def extract_contracts(code):
        contracts = []
        current_contract = []
        in_contract = False
        brace_count = 0
        lines = code.split('\n')
        for line in lines:
            stripped_line = line.strip()
            if not in_contract:
                if stripped_line.startswith(('contract ', 'interface ', 'library ')):
                    in_contract = True
                    current_contract = [line]
                    brace_count = line.count('{') - line.count('}')
            else:
                current_contract.append(line)
                brace_count += line.count('{') - line.count('}')
            if in_contract and brace_count == 0:
                contracts.append('\n'.join(current_contract).strip())
                current_contract = []
                in_contract = False
        return contracts

    def get_contract_name(contract_code):
        lines = contract_code.split('\n')
        first_line = lines[0].strip() if lines else ''
        parts = first_line.split()
        if len(parts) < 2 or parts[0] not in ['contract', 'interface', 'library']:
            return None
        name_part = parts[1]
        name = name_part.split('{')[0].split('is')[0].strip()
        return name

    # Process each contract to extract pragma and contracts
    for contract in contracts:
        code = contract.strip()
        lines = code.split('\n')
        pragma_lines = []
        in_pragma = False

        # Extract pragma lines
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('pragma'):
                pragma_lines.append(line)
                in_pragma = True
            elif in_pragma:
                break  # Stop after first non-pragma line following pragma

        # Add pragma to set
        pragma_str = '\n'.join(pragma_lines).strip()
        if pragma_str:
            pragma_set.add(pragma_str)

        # Extract rest of the code and process contracts
        rest_code = '\n'.join(lines[len(pragma_lines):]).strip()
        if rest_code:
            contracts_in_rest = extract_contracts(rest_code)
            extracted_contracts.extend(contracts_in_rest)

    # Deduplicate contracts by name, preserving order
    seen = set()
    deduped_contracts = []
    for contract in extracted_contracts:
        name = get_contract_name(contract)
        if name and name not in seen:
            seen.add(name)
            deduped_contracts.append(contract)

    # Build merged code
    merged_code = []
    # Handle pragma directives
    if len(pragma_set) == 1:
        merged_code.append(next(iter(pragma_set)))
    else:
        merged_code.extend(sorted(pragma_set))  # Sort for consistency if multiple pragmas

    # Add deduplicated contracts
    merged_code.extend(deduped_contracts)

    return '\n\n'.join(merged_code).strip()

In [ ]:
sol_dict = {}
for info in data:
    sol_dict[info['idx']] = (info['address'], info['contract'])

In [ ]:
def process_data(filepath):
    """Processes a data file and returns lists of contracts classified as 0 and 1."""
    with open(filepath, 'r') as f:
        for line in f:
            try:
                index, label = line.strip().split()
                label = int(label)
                address, contract = sol_dict[index]
                sol_dict[index] = (address, contract, label)
            except ValueError:
                print(f"Skipping invalid line: {line.strip()}")

# Example usage:
process_data('train.txt')
process_data('valid.txt')
process_data('test.txt')

In [ ]:
contract_groups = []
current_address = ""
current_group = []
for idx in range(len(sol_dict)):
    address, contract, label = sol_dict[str(idx)]
    if address != current_address:
        if current_address != "":
            contract_groups.append((current_address, current_group))
        current_address = address
        current_group = []
    current_group.append((contract, label))

In [ ]:
print(len(contract_groups))

46056


In [ ]:
peculiar_files = {}
unpeculiar_files = {}

for adr, gr in contract_groups:
    cp = 0
    contracts = [k for k, v in gr]
    for contract, label in gr:
        if label == 1:
            cp = 1
            break

    merged_sc = merge_contracts(contracts).replace(" _\n", " _;\n")
    merged_sc = remove_comments(merged_sc)
    if cp == 1:
        peculiar_files[merged_sc] = adr
    else:
        unpeculiar_files[merged_sc] = adr

In [ ]:
print(len(peculiar_files))
print(len(unpeculiar_files))

1104
44442


In [ ]:
peculiar_reentrancy_count = []
for file_content in peculiar_files.keys():
    if file_content:  # Check if file_content is not empty
        old_matches = re.findall(r"call\.value", file_content)
        new_matches = re.findall(r"call{value", file_content)
        ok = 0
        if (len(old_matches) == 1 and not new_matches) or \
           (len(new_matches) == 1 and not old_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "call.value" in line or "call{value" in line:
                  try:
                      peculiar_reentrancy_count.append(find_function(lines, i))
                  except:
                      pass

print("Number of reentrancy files with one 'call.value':", len(set(peculiar_reentrancy_count)))

Number of reentrancy files with one 'call.value': 338


In [ ]:
for k, v in peculiar_files.items():
    if "0x106b419718298f91ca576728a670597fb2e0ee4e" in v:
        print("hehe")

hehe


## Messi-Q / DeeSCVHunter

In [ ]:
reentrancy_files = os.listdir("/content/DeeSCVHunter/preprocessing/data/reentrancy/solidity_contract")
timedep_files =  os.listdir("/content/DeeSCVHunter/preprocessing/data/timestamp/solidity_contract")

def get_mapping(root, name_file, label_file):
    name_list = open(f"{root}/{name_file}", "r").readlines()
    label_list = open(f"{root}/{label_file}", "r").readlines()
    mapping = {}
    for name, label in zip(name_list, label_list):
        mapping[name.strip()] = int(label.strip())
    return mapping

reentrancy_mapping = get_mapping("/content/DeeSCVHunter/preprocessing/label/reentrancy", "reentrancy_contract_name.txt", "reentrancy_contract_label.txt")
timedep_mapping = get_mapping("/content/DeeSCVHunter/preprocessing/label/timestamp", "timestamp_contract_name.txt", "timestamp_contract_label.txt")

print(len(reentrancy_mapping))
print(len(timedep_mapping))

deescv_files = {
    "reentrancy": {},
    "timestamp": {}
}

nondeescv_files = {
    "reentrancy": {},
    "timestamp": {}
}


cnt = 0
for vul in deescv_files.keys():
    files = os.listdir(f"/content/DeeSCVHunter/preprocessing/data/{vul}/solidity_contract")
    for file in files:
        with open(f"/content/DeeSCVHunter/preprocessing/data/{vul}/solidity_contract/{file}", "r") as f:
            code = f.read()
        file_name = file.split(".")[0]
        code = remove_comments(code)
        if vul == "reentrancy":
            if reentrancy_mapping[file_name + ".sol"] == 1:
                deescv_files[vul][code] = file_name
            else:
                nondeescv_files[vul][code] = file_name
        if vul == "timestamp":
            if timedep_mapping[file_name + ".sol"] == 1:
                deescv_files[vul][code] = file_name
            else:
                nondeescv_files[vul][code] = file_name

185
185


In [ ]:
print(len(deescv_files["reentrancy"]))
print(len(deescv_files["timestamp"]))
print(len(nondeescv_files["reentrancy"]))
print(len(nondeescv_files["timestamp"]))

53
72
132
78


In [ ]:
deescv_reentrancy_count = []

for file_content, address in deescv_files["reentrancy"].items():
    if file_content:  # Check if file_content is not empty
        old_matches = re.findall(r"call\.value", file_content)
        new_matches = re.findall(r"call{ value", file_content)
        ok = 0
        if (len(old_matches) == 1 and not new_matches) or \
           (len(new_matches) == 1 and not old_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "call.value" in line:
                  try:
                      address = address + ".sol"
                      if reentrancy_mapping[address] == 1:
                          deescv_reentrancy_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

deescv_timedep_count = []

cnt = 1
for file_content, address in deescv_files["timestamp"].items():
    if file_content:  # Check if file_content is not empty
        now_matches = re.findall(r"now", file_content)
        timestamp_matches = re.findall(r"block\.timestamp", file_content)
        ok = 0
        if (len(now_matches) == 1 and not timestamp_matches) or \
           (len(timestamp_matches) == 1 and not now_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "now" in line or "block.timestamp" in line:
                  try:
                      address = address + ".sol"
                      if timedep_mapping[address] == 1:
                          deescv_timedep_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

print("Number of reentrancy files with one 'call.value':", len(set(deescv_reentrancy_count)))
print("Number of time dependency files with one 'now' or 'block.timestamp':", len(set(deescv_timedep_count)))

Number of reentrancy files with one 'call.value': 44
Number of time dependency files with one 'now' or 'block.timestamp': 11


## SolAudit Dataset

In [ ]:
# Replace with your Kaggle API credentials
!mkdir -p ~/.kaggle
!echo '{"username":"QuangNguyen711","key":"96bcd39c738542c30dcaf6019e64d695"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Replace with the dataset URL from Kaggle
!kaggle datasets download -d 'jakeclark38a/soliaudit-va-dataset-sourcecode'

Dataset URL: https://www.kaggle.com/datasets/jakeclark38a/soliaudit-va-dataset-sourcecode
License(s): unknown
  0% 0.00/38.1M [00:00<?, ?B/s]
100% 38.1M/38.1M [00:00<00:00, 902MB/s]


In [ ]:
# Unzip the downloaded dataset
!mkdir /content/soliaudit_dataset
!unzip /content/soliaudit-va-dataset-sourcecode.zip -d /content/soliaudit_dataset

Archive:  /content/soliaudit-va-dataset-sourcecode.zip
  inflating: /content/soliaudit_dataset/SoliAudit-VA-Dataset-SourceCode.csv  


In [ ]:
import pandas as pd

df = pd.read_csv("/content/soliaudit_dataset/SoliAudit-VA-Dataset-SourceCode.csv")
df

,Addr,Underflow,Overflow,CallDepth,TOD,TimeDep,Reentracy,AssertFail,CheckEffects,InlineAssembly,BlockTimestamp,LowlevelCalls,SelfDestruct,source_code
0,0x0000000000b3F879cb30FE243b4Dfee438691c04,1,1,1,0,0,0,0,1,1,0,1,0,pragma solidity ^0.4.10;\n\ncontract GasToken2...
1,0x000000002647e16d9bab9e46604d75591d289277,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
2,0x000000002bb43c83ece652d161ad0fa862129a2c,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
3,0x000000005fbe2cc9b1b684ec445caf176042348e,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
4,0x0006157838d5a6b33ab66588a6a693a57c869999,0,0,0,0,0,0,0,0,0,0,0,0,pragma solidity ^0.4.11;\n\ncontract IconomiBl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17974,0xffed1cb392e36bdf5004450ab76fab23a5337c05,1,1,0,0,0,0,0,0,0,0,1,0,pragma solidity ^0.4.4;\n\ncontract Token {\n\...
17975,0xfff18893bc6c430741b3c5ad483391a5e21220bb,0,1,0,0,0,0,1,0,0,0,0,0,pragma solidity ^0.4.13;\n\n\n/**\n * @title S...
17976,0xfff3d3b591e792eb3d937327b4b786db37ba2087,1,1,0,0,0,0,0,0,0,0,0,0,pragma solidity ^0.4.13;\ncontract owned {\n ...
17977,0xfffce2dc587badbd10b4fe17f0f5f293458f6793,0,1,0,0,0,0,1,1,0,1,0,0,pragma solidity ^0.4.18;\n\ncontract Admin {\n...


In [ ]:
soliaudit_reentrancy_files = {}
soliaudit_timedep_files = {}
soliaudit_non_timedep_files = {}

for index, row in df.iterrows():
    sc = remove_comments(row['source_code'])
    if row['Reentracy'] == 1:
        # soliaudit_reentrancy_files.append((row['source_code'], row['source_code']))
        soliaudit_reentrancy_files[sc] = row['Addr']
    if row['BlockTimestamp'] == 1:
        # soliaudit_timedep_files.append((row['source_code'], row['source_code']))
        soliaudit_timedep_files[sc] = row['Addr']
    if row['TimeDep'] == 1:
        # soliaudit_timedep_files.append((row['source_code'], row['source_code']))
        soliaudit_timedep_files[sc] = row['Addr']
    if row['TimeDep'] == 0 and row['BlockTimestamp'] == 0:
        soliaudit_non_timedep_files[sc] = row['Addr']


print("Reentrancy files:", len(soliaudit_reentrancy_files))
print("Time Dependency files:", len(soliaudit_timedep_files))
print("Non Time Dependency files:", len(soliaudit_non_timedep_files))

Reentrancy files: 439
Time Dependency files: 4768
Non Time Dependency files: 11172


In [ ]:
from types import new_class
# prompt: In reentrancy_files check how many files have only one "call.value" and check in timedep_files check how many files have only one "now" or one "block.timestamp"

import re

soliaudit_timedep_count = []
for file_content, adr in soliaudit_timedep_files.items():
    if file_content:  # Check if file_content is not empty
        now_matches = re.findall(r"now", file_content)
        timestamp_matches = re.findall(r"block\.timestamp", file_content)
        ok = 0
        if (len(now_matches) == 1 and not timestamp_matches) or \
           (len(timestamp_matches) == 1 and not now_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "now" in line or "block.timestamp" in line:
                  try:
                      soliaudit_timedep_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

print("Number of time dependency files with one 'now' or 'block.timestamp':", len(set(soliaudit_timedep_count)))

Number of time dependency files with one 'now' or 'block.timestamp': 454


## Cleaning

In [ ]:
def sort_dict_by_key(input_dict):
  """Sorts a dictionary by its keys and returns the sorted dictionary.

  Args:
    input_dict: The dictionary to be sorted.

  Returns:
    A new dictionary with the same key-value pairs as the input dictionary,
    but sorted by keys.
  """
  return dict(sorted(input_dict.items()))


In [ ]:
path_smartbugs_wild = "ge-sc/data/smartbugs-wild-clean-contracts"

path_save_non_reentrancy = "/content/non_reentrancy"
path_save_non_timestamp =  "/content/non_timestamp"

flags_reentrancy = ["call.value", "call{ value", "call{value"]
flags_timestamp = ["block.timestamp", "now"]

In [ ]:
# @title Utils function
import re
import json
from pathlib import Path

def read_file_sol(path):
  with open(path, "r") as file:
    data = file.read()
    data = remove_comments(data)
  return data

def check_source_with_flag(source, flag):
  for item in flag:
    if item in source: return True
  return False

def verify_syntax(files, flags):
  print(f"Starting verify syntax with {flags}")
  list_file_error = []
  for idx, filel in enumerate(files):
    print(f"Verify syntax file: {idx} - {filel}")
    source = read_file_sol(filel)
    if check_source_with_flag(source, flags):
      list_file_error.append(filel)
  return list_file_error

In [ ]:
import glob
path_file_sol = glob.glob(path_smartbugs_wild + "/*.sol", recursive=True)

# verify syntax
sol_file_reentrancy = verify_syntax(path_file_sol, flags_reentrancy)
sol_file_reentrancy = sorted(sol_file_reentrancy)
sol_file_timestamp = verify_syntax(path_file_sol, flags_timestamp)
sol_file_timestamp = sorted(sol_file_timestamp)


print(len(sol_file_reentrancy))
print(len(sol_file_timestamp))

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Verify syntax file: 487 - ge-sc/data/smartbugs-wild-clean-contracts/0x8c1e0e4bfeecd856bebbe7e0a6740df849bbb9a8.sol
Verify syntax file: 488 - ge-sc/data/smartbugs-wild-clean-contracts/0xa3474fe420ba2c8190ba03365afba03b478ecf12.sol
Verify syntax file: 489 - ge-sc/data/smartbugs-wild-clean-contracts/0x395a200ab37e622ad721f6c0d2fa913b522c84cf.sol
Verify syntax file: 490 - ge-sc/data/smartbugs-wild-clean-contracts/0x6682195e2a0048ce38b727a3711802d58244606e.sol
Verify syntax file: 491 - ge-sc/data/smartbugs-wild-clean-contracts/0xd2299b3098cf5e13144caebfdad61ebe505233dc.sol
Verify syntax file: 492 - ge-sc/data/smartbugs-wild-clean-contracts/0x117b4ff6e29baa41592fbc02abd2c0b40110d6e6.sol
Verify syntax file: 493 - ge-sc/data/smartbugs-wild-clean-contracts/0xb7cfd7f79aa6efb213daf5523c25e4bce11e1037.sol
Verify syntax file: 494 - ge-sc/data/smartbugs-wild-clean-contracts/0x27838124dbf87421e33867d46f5060170a5cb8ed.sol
Verify syntax file: 495

In [ ]:
mando_non_reentrancy_syntax_files = {}
mando_non_timestamp_syntax_files = {}

for link in sol_file_reentrancy:
    with open(link, "r") as f:
        code = f.read()
    code = remove_comments(code)
    name = Path(link).stem
    mando_non_reentrancy_syntax_files[code] = name

for link in sol_file_timestamp:
    with open(link, "r") as f:
        code = f.read()
    code = remove_comments(code)
    name = Path(link).stem
    mando_non_timestamp_syntax_files[code] = name

In [ ]:
mando_non_reentrancy_syntax_files = sort_dict_by_key(mando_non_reentrancy_syntax_files)
mando_non_timestamp_syntax_files = sort_dict_by_key(mando_non_timestamp_syntax_files)

In [ ]:
print(len(mando_non_reentrancy_syntax_files))
print(len(mando_non_timestamp_syntax_files))

17
582


In [ ]:
total_reentrancy_funcs = list(set(smartbugs_functions["reentrancy"]["vulnerable"])) + list(set(peculiar_reentrancy_count)) + list(set(deescv_reentrancy_count))
undup_reentrancy_funcs = list(set(total_reentrancy_funcs))
total_reentrancy_funcs = sorted(total_reentrancy_funcs)
undup_reentrancy_funcs = sorted(undup_reentrancy_funcs)
print(len(total_reentrancy_funcs))
print(len(undup_reentrancy_funcs))
total_timedep_funcs = list(set(smartbugs_functions["time_manipulation"]["vulnerable"])) + list(set(deescv_timedep_count)) + list(set(soliaudit_timedep_count))
undup_timedep_funcs = list(set(total_timedep_funcs))
total_timedep_funcs = sorted(total_timedep_funcs)
undup_timedep_funcs = sorted(undup_timedep_funcs)
print(len(total_timedep_funcs))
print(len(undup_timedep_funcs))

total_reentrancy_code = {}
for sc, adr in smartbugs_files["reentrancy"].items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr
for sc, adr in peculiar_files.items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr
for sc, adr in deescv_files["reentrancy"].items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr

total_timedep_code = {}
for sc, adr in smartbugs_files["time_manipulation"].items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr
for sc, adr in deescv_files["timestamp"].items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr
for sc, adr in soliaudit_timedep_files.items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr

396
396
471
467


In [ ]:
total_reentrancy_code = sort_dict_by_key(total_reentrancy_code)
total_timedep_code = sort_dict_by_key(total_timedep_code)

In [ ]:
for i in undup_reentrancy_funcs[:10]:
    print(i.splitlines(keepends=True)[0].strip())
    print("-" * 100)

function jackpotSend() payable public {
----------------------------------------------------------------------------------------------------
function actualTransfer (address payable from, address payable to, uint value, bytes memory data, string memory func, bool careAboutHumanity) internal{
----------------------------------------------------------------------------------------------------
function executeTransaction(uint transactionId)
----------------------------------------------------------------------------------------------------
function finish()
----------------------------------------------------------------------------------------------------
function mintETHRewards( address _contract, uint256 _amount ) public onlyManager() {
----------------------------------------------------------------------------------------------------
function send(address _to, uint _value, bytes _data) only_owner {
--------------------------------------------------------------------------------------

In [ ]:
for i in list(total_reentrancy_code.keys())[:10]:
    print(i.splitlines(keepends=True)[0].strip())
    print("-" * 100)

contract Accrual_account
----------------------------------------------------------------------------------------------------
contract AmIOnTheFork {
----------------------------------------------------------------------------------------------------
contract Attack {
----------------------------------------------------------------------------------------------------
contract EtherStore {
----------------------------------------------------------------------------------------------------
contract Etheropt {
----------------------------------------------------------------------------------------------------
contract Factory {
----------------------------------------------------------------------------------------------------
contract Factory {
----------------------------------------------------------------------------------------------------
contract HongConfiguration {
----------------------------------------------------------------------------------------------------
contract LooneyL

In [ ]:
for i in total_timedep_funcs[:10]:
    print(i.splitlines(keepends=True)[0].strip())
    print("-" * 100)

constructor() public payable {}
----------------------------------------------------------------------------------------------------
function ARBITRAGECrowdsale(address _walletOwner, address _partnerHandler) public {
----------------------------------------------------------------------------------------------------
function Add(uint _version, string _name, string _hash) {
----------------------------------------------------------------------------------------------------
function AddMessage(address _adr,uint _val,string _data)
----------------------------------------------------------------------------------------------------
function AddOwnership(string _btcAddress, uint _verifyCode, string _referCode) isActive public returns(ResultCode) {
----------------------------------------------------------------------------------------------------
function AddOwnership(string _btcAddress, uint _verifyCode, string _referCode) isActive public returns(ResultCode) {
------------------------------

In [ ]:
for i in list(total_timedep_code.keys())[:10]:
    print(i.splitlines(keepends=True)[0].strip())
    print("-" * 100)

/* Simple token - simple token for PreICO and ICO
----------------------------------------------------------------------------------------------------
/* Token - simple token for PreICO and ICO
----------------------------------------------------------------------------------------------------
/* Token - simple token for PreICO and ICO
----------------------------------------------------------------------------------------------------
contract AbstractStarbaseCrowdsale {
----------------------------------------------------------------------------------------------------
contract BCFBaseCompetition {
----------------------------------------------------------------------------------------------------
contract BCFSafe {
----------------------------------------------------------------------------------------------------
contract BaseRegistry {
----------------------------------------------------------------------------------------------------
contract BatLimitAsk{
-------------------------

In [ ]:
print(len(total_reentrancy_code))
print(len(total_timedep_code))

1131
4845


### Get Re-entrancy Dataset

In [ ]:
import random

best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    # Assuming undup_reentrancy_funcs is defined in the previous code
    # Split undup_reentrancy_funcs into train and test sets
    shuffled_funcs = undup_reentrancy_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    reen_train_vul_funcs = shuffled_funcs[:split_index]
    reen_test_vul_funcs = shuffled_funcs[split_index:]

    # print("Number of train functions:", len(reen_train_vul_funcs))
    # print("Number of test functions:", len(reen_test_vul_funcs))

    # Count occurrences of train functions in total_reentrancy_code
    reen_train_vul_count = []
    for code, adr in total_reentrancy_code.items():
        cp = 0
        for func in reen_train_vul_funcs:
            if func in code:
                cp = 1
                break
        for func2 in reen_test_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_train_vul_count.append((adr, code))

    # Count occurrences of test functions in total_reentrancy_code
    reen_test_vul_count = []
    for code, adr in total_reentrancy_code.items():
        cp = 0
        for func in reen_test_vul_funcs:
            if func in code and (adr, code) not in reen_train_vul_count:
                cp = 1
                break
        for func2 in reen_train_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_test_vul_count.append((adr, code))

    if len(reen_train_vul_count) / len(reen_test_vul_count) >= 3.8 and len(reen_train_vul_count) / len(reen_test_vul_count) <= 4.2:
        if len(reen_train_vul_count) + len(reen_test_vul_count) > best_preserve:
            best_preserve = len(reen_train_vul_count) + len(reen_test_vul_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(reen_train_vul_count) / len(reen_test_vul_count), 2)} with able to retain {len(reen_train_vul_count) + len(reen_test_vul_count)} out of {len(total_reentrancy_code)}")
        # break'

Seed 8 is compatible ~3.96 with able to retain 942 out of 1131
Seed 9 is compatible ~3.99 with able to retain 948 out of 1131
Seed 15 is compatible ~4.2 with able to retain 910 out of 1131
Seed 17 is compatible ~3.91 with able to retain 894 out of 1131
Seed 20 is compatible ~3.86 with able to retain 933 out of 1131
Seed 29 is compatible ~4.17 with able to retain 915 out of 1131
Seed 30 is compatible ~3.83 with able to retain 908 out of 1131
Seed 35 is compatible ~3.88 with able to retain 941 out of 1131
Seed 41 is compatible ~3.8 with able to retain 927 out of 1131
Seed 48 is compatible ~3.9 with able to retain 940 out of 1131
Seed 49 is compatible ~3.95 with able to retain 910 out of 1131
Seed 50 is compatible ~4.04 with able to retain 928 out of 1131
Seed 58 is compatible ~4.12 with able to retain 943 out of 1131
Seed 70 is compatible ~3.84 with able to retain 906 out of 1131
Seed 75 is compatible ~4.16 with able to retain 939 out of 1131
Seed 77 is compatible ~4.12 with able to reta

In [ ]:
print(best_seed)

85


In [ ]:
def get_filtered_data(data_dict, data_num):
    filtered_data = []
    i = 0
    step = 0
    key_list = list(data_dict.keys())
    while len(filtered_data) < data_num:
        if i > len(key_list) - 1:
            i = 0
            step += 1
        if step < len(data_dict[key_list[i]]):
            filtered_data.append(data_dict[key_list[i]][step])
        i += 1
    return filtered_data

In [ ]:
import random

random.seed(best_seed)
# Assuming undup_reentrancy_funcs is defined in the previous code
# Split undup_reentrancy_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------------------
shuffled_funcs = undup_reentrancy_funcs.copy()
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
reen_train_vul_funcs = shuffled_funcs[:split_index]
reen_test_vul_funcs = shuffled_funcs[split_index:]

print("Number of train functions:", len(reen_train_vul_funcs))
print("Number of test functions:", len(reen_test_vul_funcs))

# Check occurrences of vul train functions in total_reentrancy_code -----------------------------------------------------------------------------------------------------------------
reen_train_vul_count = []
reen_train_vul_dict = {}

for code, adr in total_reentrancy_code.items():
    cp = 0
    for func in reen_train_vul_funcs:
        if func in code:
            cp = 1
            break
    for func2 in reen_test_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_train_vul_count.append((adr, code))
        if func not in reen_train_vul_dict:
            reen_train_vul_dict[func] = []
        reen_train_vul_dict[func].append((adr, code))

# Check occurrences of vul test functions in total_reentrancy_code -----------------------------------------------------------------------------------------------------------------
reen_test_vul_count = []
reen_test_vul_dict = {}

for code, adr in total_reentrancy_code.items():
    cp = 0
    for func in reen_test_vul_funcs:
        if func in code and (adr, code) not in reen_train_vul_count:
            cp = 1
            break
    for func2 in reen_train_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_test_vul_count.append((adr, code))
        if func not in reen_test_vul_dict:
            reen_test_vul_dict[func] = []
        reen_test_vul_dict[func].append((adr, code))

# Check number of reentrancy vul source codes that have more than 1 call.value -----------------------------------------------------------------------------------------------------
reen_vul_not_counted = []

for code, adr in total_reentrancy_code.items():
    if (adr, code) not in reen_train_vul_count and (adr, code) not in reen_test_vul_count:
        reen_vul_not_counted.append((adr, code))

random.shuffle(reen_vul_not_counted)
split_index = int(0.8 * len(reen_vul_not_counted))
reen_train_vul_not_counted = reen_vul_not_counted[:split_index]
reen_test_vul_not_counted = reen_vul_not_counted[split_index:]

# reen_train_vul_count = get_filtered_data(reen_train_vul_dict, )

print("Number of vulnerability source codes with 1 call.value containing train functions:", len(reen_train_vul_count))
print("Number of vulnerability source codes with 1 call.value containing test functions:", len(reen_test_vul_count))
print("Number of vulnerability source codes with more than 1 call.value containing train functions:", len(reen_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 call.value containing test functions:", len(reen_test_vul_not_counted))

# Get all source code that doesn't contain reentrancy in puculiar dataset --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

unpeculiar_syntax_files = []
unpeculiar_not_syntax_files = []

unpeculiar_files = sort_dict_by_key(unpeculiar_files)
nondeescv_files['reentrancy'] = sort_dict_by_key(nondeescv_files['reentrancy'])
mando_non_reentrancy_syntax_files = sort_dict_by_key(mando_non_reentrancy_syntax_files)

for sc, adr in unpeculiar_files.items():
    if "call.value" in sc:
        unpeculiar_syntax_files.append((adr, sc))
    else:
        unpeculiar_not_syntax_files.append((adr, sc))

for sc, adr in nondeescv_files["reentrancy"].items():
    if "call.value" in sc:
        unpeculiar_syntax_files.append((adr, sc))
    else:
        unpeculiar_not_syntax_files.append((adr, sc))

for sc, adr in mando_non_reentrancy_syntax_files.items():
    if "call.value" in sc:
        unpeculiar_syntax_files.append((adr, sc))

# Get non-vulnerability functions still contain syntax call.value and functions that not contain syntax call.value ----------------------------------------------------------------------------
reen_non_vul_syntax_funcs = []
reen_non_vul_not_syntax_funcs = []

check_source_code_syntax_num = []

for adr, sc in unpeculiar_syntax_files:
    lines = sc.splitlines(keepends=True)
    cp = 0
    for i, line in enumerate(lines):
        if "function " in line or "modifier " in line or "constructor " in line:
            cp = 1
            try:
                sampled_func = find_function(lines, i)
                if "call.value" in sampled_func:
                    reen_non_vul_syntax_funcs.append(sampled_func)
                else:
                    reen_non_vul_not_syntax_funcs.append(sampled_func)
            except:
                pass
    if cp == 1:
        check_source_code_syntax_num.append(adr)

print("Number of non-vulnerability source codes containing syntax call.value:", len(check_source_code_syntax_num))

reen_non_vul_syntax_funcs = list(set(reen_non_vul_syntax_funcs))
reen_non_vul_not_syntax_funcs = list(set(reen_non_vul_not_syntax_funcs))

print("Number of non-vulnerability functions containing syntax call.value:", len(reen_non_vul_syntax_funcs))
print("Number of non-vulnerability functions not containing syntax call.value:", len(reen_non_vul_not_syntax_funcs))
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Number of train functions: 316
Number of test functions: 80
Number of vulnerability source codes with 1 call.value containing train functions: 765
Number of vulnerability source codes with 1 call.value containing test functions: 184
Number of vulnerability source codes with more than 1 call.value containing train functions: 145
Number of vulnerability source codes with more than 1 call.value containing test functions: 37
Number of non-vulnerability source codes containing syntax call.value: 554
Number of non-vulnerability functions containing syntax call.value: 334
Number of non-vulnerability functions not containing syntax call.value: 9826


In [ ]:
reen_non_vul_syntax_funcs = sorted(reen_non_vul_syntax_funcs)
reen_non_vul_not_syntax_funcs = sorted(reen_non_vul_not_syntax_funcs)
unpeculiar_syntax_files = sorted(unpeculiar_syntax_files)

In [ ]:
best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    shuffled_funcs = reen_non_vul_syntax_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    reen_train_non_vul_syntax_funcs = shuffled_funcs[:split_index]
    reen_test_non_vul_syntax_funcs = shuffled_funcs[split_index:]

    reen_train_non_vul_syntax_count = []
    reen_test_non_vul_syntax_count = []


    # Count occurrences of train functions in total_reentrancy_code
    for adr, code in unpeculiar_syntax_files:
        cp = 0
        for func in reen_train_non_vul_syntax_funcs:
            if func in code:
                cp = 1
                break
        for func2 in reen_test_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_train_non_vul_syntax_count.append((adr, code))


    # Count occurrences of test functions in total_reentrancy_code
    for adr, code in unpeculiar_syntax_files:
        cp = 0
        for func in reen_test_non_vul_syntax_funcs:
            if func in code and (adr, code) not in reen_train_non_vul_syntax_count:
                cp = 1
                break
        for func2 in reen_train_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_test_non_vul_syntax_count.append((adr, code))


    # print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
    # print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

    if len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count) >= 3.8 and len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count) <= 4.2:
        if len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count) > best_preserve:
            best_preserve = len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count), 2)} with able to retain {len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count)} out of {len(unpeculiar_syntax_files)}")
        # break

Seed 0 is compatible ~4.18 with able to retain 368 out of 554
Seed 16 is compatible ~3.99 with able to retain 379 out of 554
Seed 19 is compatible ~3.89 with able to retain 367 out of 554
Seed 20 is compatible ~3.91 with able to retain 373 out of 554
Seed 27 is compatible ~4.11 with able to retain 363 out of 554
Seed 44 is compatible ~3.84 with able to retain 358 out of 554
Seed 50 is compatible ~3.82 with able to retain 376 out of 554
Seed 58 is compatible ~3.8 with able to retain 365 out of 554
Seed 63 is compatible ~4.15 with able to retain 381 out of 554
Seed 68 is compatible ~3.82 with able to retain 366 out of 554
Seed 75 is compatible ~4.07 with able to retain 370 out of 554
Seed 82 is compatible ~4.1 with able to retain 372 out of 554
Seed 83 is compatible ~4.15 with able to retain 376 out of 554
Seed 84 is compatible ~4.05 with able to retain 374 out of 554
Seed 85 is compatible ~4.17 with able to retain 372 out of 554
Seed 86 is compatible ~3.88 with able to retain 376 out of

In [ ]:
print(best_seed)

63


In [ ]:
random.seed(best_seed)
shuffled_funcs = reen_non_vul_syntax_funcs.copy()
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
reen_train_non_vul_syntax_funcs = shuffled_funcs[:split_index]
reen_test_non_vul_syntax_funcs = shuffled_funcs[split_index:]

# reen_sampled_not_syntax_funcs = random.sample(reen_non_vul_not_syntax_funcs, 5000)
# random.shuffle(reen_sampled_not_syntax_funcs)
# split_index = int(0.8 * len(reen_sampled_not_syntax_funcs))
# reen_train_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[:split_index]
# reen_test_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[split_index:]
print(len(reen_non_vul_not_syntax_funcs))

reen_train_non_vul_syntax_count, reen_test_non_vul_syntax_count = [], []
reen_train_non_vul_syntax_dict, reen_test_non_vul_syntax_dict = {}, {}

# Check occurrences of non-vul train functions in unpeculiar_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in unpeculiar_syntax_files:
    cp = 0
    for func in reen_train_non_vul_syntax_funcs:
        if func in code:
            cp = 1
            break
    for func2 in reen_test_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_train_non_vul_syntax_count.append((adr, code))
        if func not in reen_train_non_vul_syntax_dict:
            reen_train_non_vul_syntax_dict[func] = []
        reen_train_non_vul_syntax_dict[func].append((adr, code))

# Check occurrences of non-vul test functions in unpeculiar_syntax_files ------------------------------------------------------------------------------------------------------------------
for adr, code in unpeculiar_syntax_files:
    cp = 0
    for func in reen_test_non_vul_syntax_funcs:
        if func in code and (adr, code) not in reen_train_non_vul_syntax_count:
            cp = 1
            break
    for func2 in reen_train_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_test_non_vul_syntax_count.append((adr, code))
        if func not in reen_test_non_vul_syntax_dict:
            reen_test_non_vul_syntax_dict[func] = []
        reen_test_non_vul_syntax_dict[func].append((adr, code))

print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

9826
Number of non-vulnerability source codes contain call.value in train: 307
Number of non-vulnerability source codes contain call.value in test: 74


In [ ]:
random.seed(63)

In [ ]:
tmp1, tmp2 = [], []
for func in reen_non_vul_not_syntax_funcs:
    train_sc = [k[1] for k in reen_train_non_vul_syntax_count]
    test_sc = [k[1] for k in reen_test_non_vul_syntax_count]
    cpa = 0
    cpb = 0
    for k in train_sc:
        if func in k:
            cpa = 1
            break
    for k in test_sc:
        if func in k:
            cpb = 1
            break
    if cpa == 1 and cpb == 0:
        tmp1.append(func)
    elif cpa == 0 and cpb == 1:
        tmp2.append(func)

print(len(tmp1))
print(len(tmp2))

# 5653
# 1335

5653
1335


In [ ]:
# reen_sampled_not_syntax_funcs = random.sample(tmp1, 5000)
# random.shuffle(reen_sampled_not_syntax_funcs)
# split_index = int(0.8 * len(reen_sampled_not_syntax_funcs))
reen_train_non_vul_not_syntax_funcs = random.sample(tmp1, int(5000 * 0.8))
reen_test_non_vul_not_syntax_funcs = random.sample(tmp2, int(5000 * 0.2))

In [ ]:
print("Number of vulnerability source codes with 1 call.value containing train functions:", len(reen_train_vul_count))
print("Number of vulnerability source codes with 1 call.value containing test functions:", len(reen_test_vul_count))

print("Number of vulnerability source codes with more than 1 call.value containing train functions:", len(reen_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 call.value containing test functions:", len(reen_test_vul_not_counted))

print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

Number of vulnerability source codes with 1 call.value containing train functions: 765
Number of vulnerability source codes with 1 call.value containing test functions: 184
Number of vulnerability source codes with more than 1 call.value containing train functions: 145
Number of vulnerability source codes with more than 1 call.value containing test functions: 37
Number of non-vulnerability source codes contain call.value in train: 307
Number of non-vulnerability source codes contain call.value in test: 74


In [ ]:
reen_train_vul_total = get_filtered_data(reen_train_vul_dict, int(len(reen_train_non_vul_syntax_count) * 2/3))
reen_train_vul_total += random.sample(reen_train_vul_not_counted, int(len(reen_train_non_vul_syntax_count) * 1/3))
reen_test_vul_total = get_filtered_data(reen_test_vul_dict, int(len(reen_test_non_vul_syntax_count) * 2/3)) + random.sample(reen_test_vul_not_counted, int(len(reen_test_non_vul_syntax_count) * 1/3))
reen_train_non_vul_total = reen_train_non_vul_syntax_count
reen_test_non_vul_total = reen_test_non_vul_syntax_count

print("Number of vulnerability source codes in train:", len(reen_train_vul_total))
print("Number of vulnerability source codes in test:", len(reen_test_vul_total))
print("Number of non-vulnerability source codes in train:", len(reen_train_non_vul_total))
print("Number of non-vulnerability source codes in test:", len(reen_test_non_vul_total))

Number of vulnerability source codes in train: 306
Number of vulnerability source codes in test: 73
Number of non-vulnerability source codes in train: 307
Number of non-vulnerability source codes in test: 74


In [ ]:
cnt = 0
for adr, src in reen_train_non_vul_total:
    if "call.value" in src:
        cnt += 1

print(cnt)

307


In [ ]:
cnt = 0
for adr, src in reen_train_non_vul_total:
    if "call.value" in remove_comments(src):
        cnt += 1

print(cnt)

307


In [ ]:
cnt = 0
for adr, src in reen_train_non_vul_total:
    if "call.value" in clean_code_formatting(remove_comments(src)):
        cnt += 1

print(cnt)

307


In [ ]:
reen_train_non_vul_funcs = reen_train_non_vul_syntax_funcs + reen_train_non_vul_not_syntax_funcs
reen_test_non_vul_funcs = reen_test_non_vul_syntax_funcs + reen_test_non_vul_not_syntax_funcs

print("Number of non-vulnerability functions in train:", len(reen_train_non_vul_funcs))
print("Number of non-vulnerability functions in test:", len(reen_test_non_vul_funcs))

Number of non-vulnerability functions in train: 4267
Number of non-vulnerability functions in test: 1067


In [ ]:
print("Number of non-vulnerability functions in train:", len(reen_train_vul_funcs))
print("Number of non-vulnerability functions in test:", len(reen_test_vul_funcs))

Number of non-vulnerability functions in train: 316
Number of non-vulnerability functions in test: 80


In [ ]:
for k in reen_train_vul_total[:10]:
    print(k[0])

21390
Bank_attack
EtherStore
0x64a51dc1b6d914e6dc59e99bb6cad46e208c7358
0x41e3bcaa7d7662dd768c1510014de7d5058183dc
0x22a97c80d7e0a9ae616737e3b8b531248f4ef91d
0x4aea7cf559f67cedcad07e12ae6bc00f07e8cf65
0xaf30d2a7e90d7dc361c8c4585e9bb7d2f6f15bc7
0x70396c561849d1ed656c6e0d2a04b83131c0c645
0x37dad2f4f477b085c5e91a10d6a5eab6dab6a445


In [ ]:
for k in reen_test_vul_total[:10]:
    print(k[0])

0x312802f59528256ca45da2b5dbb365ec4d037398
dumbDAO
0x63c66a8afb5041ca4abe5bc0d1a50bcdfb72c159
0x0adf73d077646177ed942330cb062b44282abf9e
0x7a46c781b593068d5e987b191e9c2f7413e22aee
0x522b0f328ca716b5b676cab767372d48853ff040
0x0f17caa887cd584ebe662f15f48545085142304e
0xbece02eea2e769674404791d5b8984898df45459
0xee62b5ed03fff475499dc1d20a7a9fb08fa78574
0x539da201f33a25e4a782d3b42eb0f0a83c0fd753


In [ ]:
for k in reen_train_non_vul_total[:10]:
    print(k[0])

0x00888096c1cdeb35bb3772f9080227aa6c9968ad
0x014324b2307b1a578e3d64aa36e6e4b22b118061
0x02cc75f4f0e29cdc76a94ec38fb5281f83c9d93f
0x047a2c85394dd7f475027fc2bf1753babb454094
0x06df142f76ae6efb9025779f58f1e109dce402f0
0x07307d0b136a79bac718f43388aed706389c4588
0x09e0c54ed4cffca45d691d5eb7b976d650f5904c
0x09f5f9413cefd61044db02940540155507bdcc55
0x0b444993e305016f213a030c9af4013a8c537b63
0x0d945b31cb2fa8d6d9bf1bb2a5d603435922815e


In [ ]:
for k in reen_test_non_vul_total[:10]:
    print(k[0])

0x073c4ea5a89d0e5aa0408a1b6dd8bbfcdba985c0
0x1e5028fcb334d79633f5a6665347d2eaa02cb406
0x240f9f86b1465bf1b8eb29bc88cbf65573dfdd97
0x347e94e12c623d7b9d51b3f143ff42b73d619773
0x3875151e877cb7c048d9b8f5045debf46babe02b
0x4566d68ea96fc2213f2446f0dd0f482146cee96d
0x461733c17b0755ca5649b6db08b3e213fcf22546
0x4b9a0d1d725b91c47729d35e3dd174179891cc6c
0x4f416e928a2cf93d90772c9d8e070a8b1f5b3f36
0x522b0f328ca716b5b676cab767372d48853ff040


In [ ]:
for f in reen_train_non_vul_funcs[:10]:
    print(f.splitlines(keepends=True)[0].strip())
    print("-" * 100)

function crowdsale() public payable returns (bool) {
----------------------------------------------------------------------------------------------------
function approve(address spender, uint256 value) public returns (bool);
----------------------------------------------------------------------------------------------------
function executeTransaction(uint transactionId) public notExecuted(transactionId) {
----------------------------------------------------------------------------------------------------
function transferFrom(address, address, uint) public returns (bool);
----------------------------------------------------------------------------------------------------
function sendEther(address _to) external payable onlyOwner {
----------------------------------------------------------------------------------------------------
function takeEtherProfits(){
----------------------------------------------------------------------------------------------------
function buyWithAddress(ad

In [ ]:
for f in reen_train_vul_funcs[:10]:
    print(f.splitlines(keepends=True)[0].strip())
    print("-" * 100)

function executeTransaction(uint transactionId)
----------------------------------------------------------------------------------------------------
function distributeExternal(uint256 _rID, uint256 _pID, uint256 _eth, uint256 _affID, uint256 _team, F3Ddatasets.EventReturns memory _eventData_)
----------------------------------------------------------------------------------------------------
function collect() onlyOwner {
----------------------------------------------------------------------------------------------------
function executeTransaction(uint transactionId)
----------------------------------------------------------------------------------------------------
function payFund() payable public {
----------------------------------------------------------------------------------------------------
function distributeExternal(uint256 _rID, uint256 _pID, uint256 _eth, uint256 _affID, uint256 _team, F3Ddatasets.EventReturns memory _eventData_)
----------------------------------------

### Get Timestamp Dependency Dataset

In [ ]:
import random

best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    # Assuming undup_timedep_funcs is defined in the previous code
    # Split undup_timedep_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------
    shuffled_funcs = undup_timedep_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    time_train_vul_funcs = shuffled_funcs[:split_index]
    time_test_vul_funcs = shuffled_funcs[split_index:]

    # Count occurrences of train functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
    time_train_vul_count = []
    for code, adr in total_timedep_code.items():
        cp = 0
        for func in time_train_vul_funcs:
            if func in code:
                cp = 1
                break
        for func2 in time_test_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_train_vul_count.append((adr, code))

    # Count occurrences of test functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
    time_test_vul_count = []
    for code, adr in total_timedep_code.items():
        cp = 0
        for func in time_test_vul_funcs:
            if func in code and (adr, code) not in time_train_vul_count:
                cp = 1
                break
        for func2 in time_train_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_test_vul_count.append((adr, code))

    if len(time_train_vul_count) / len(time_test_vul_count) >= 3.6 and len(time_train_vul_count) / len(time_test_vul_count) <= 4.2:
        if len(reen_train_vul_count) + len(reen_test_vul_count) > best_preserve:
            best_preserve = len(reen_train_vul_count) + len(reen_test_vul_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(time_train_vul_count) / len(time_test_vul_count), 2)} with able to retain {len(time_train_vul_count) + len(time_test_vul_count)} out of {len(total_timedep_code)}")
        # break

Seed 11 is compatible ~4.08 with able to retain 2179 out of 4845
Seed 87 is compatible ~4.01 with able to retain 2151 out of 4845
Seed 93 is compatible ~4.14 with able to retain 2153 out of 4845


In [ ]:
print(best_seed)

11


In [ ]:
soliaudit_non_timedep_files = sort_dict_by_key(soliaudit_non_timedep_files)
nondeescv_files["timestamp"] = sort_dict_by_key(nondeescv_files["timestamp"])
mando_non_timestamp_syntax_files = sort_dict_by_key(mando_non_timestamp_syntax_files)

In [ ]:
from datetime import time
import random

random.seed(best_seed)
# Assuming undup_timedep_funcs is defined in the previous code
# Split undup_timedep_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------
shuffled_funcs = undup_timedep_funcs.copy()
print("Number of vulnerability functions:", len(shuffled_funcs))
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
time_train_vul_funcs = shuffled_funcs[:split_index]
time_test_vul_funcs = shuffled_funcs[split_index:]

print("Number of train functions:", len(time_train_vul_funcs))
print("Number of test functions:", len(time_test_vul_funcs))

print(len(total_timedep_code))
# Count occurrences of vul train functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
time_train_vul_count = []
time_train_vul_dict = {}

for code, adr in total_timedep_code.items():
    cp = 0
    for func in time_train_vul_funcs:
        if func in code:
            cp = 1
            break
    for func2 in time_test_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_train_vul_count.append((adr, code))
        if func not in time_train_vul_dict:
            time_train_vul_dict[func] = []
        time_train_vul_dict[func].append((adr, code))

# Count occurrences of vul test functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
time_test_vul_count = []
time_test_vul_dict = {}

for code, adr in total_timedep_code.items():
    cp = 0
    for func in time_test_vul_funcs:
        if func in code and (adr, code) not in time_train_vul_count:
            cp = 1
            break
    for func2 in time_train_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_test_vul_count.append((adr, code))
        if func not in time_test_vul_dict:
            time_test_vul_dict[func] = []
        time_test_vul_dict[func].append((adr, code))

# Check number of timestamp vul source codes that have more than 1 block.timestamp or now -----------------------------------------------------------------------------------------------------
time_vul_not_counted = []

for code, adr in total_timedep_code.items():
    if (adr, code) not in time_train_vul_count and (adr, code) not in time_test_vul_count:
        time_vul_not_counted.append((adr, code))

random.shuffle(time_vul_not_counted)
split_index = int(0.8 * len(time_vul_not_counted))
time_train_vul_not_counted = time_vul_not_counted[:split_index]
time_test_vul_not_counted = time_vul_not_counted[split_index:]

print("Number of vulnerability source codes with 1 block.timestamp or 1 now containing train functions:", len(time_train_vul_count))
print("Number of vulnerability source codes with 1 block.timestamp or 1 now containing test functions:", len(time_test_vul_count))
print("Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions:", len(time_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions:", len(time_test_vul_not_counted))
# Get all source code that doesn't contain timestamp dependency in puculiar dataset --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

soliaudit_syntax_files = []
soliaudit_not_syntax_files = []

for sc, adr in soliaudit_non_timedep_files.items():
    if "block.timestamp" in sc or "now" in sc:
        soliaudit_syntax_files.append((adr, sc))
    else:
        soliaudit_not_syntax_files.append((adr, sc))

for sc, adr in nondeescv_files["timestamp"].items():
    if "block.timestamp" in sc or "now" in sc:
        soliaudit_syntax_files.append((adr, sc))
    else:
        soliaudit_not_syntax_files.append((adr, sc))

for sc, adr in mando_non_timestamp_syntax_files.items():
    if "block.timestamp" in sc or "now" in sc:
        soliaudit_syntax_files.append((adr, sc))

# Get non-vulnerability functions still contain syntax (block.timestamp or now) and functions that not contain syntax (block.timestamp or now) ----------------------------------------------------------------------------
time_non_vul_syntax_funcs = []
time_non_vul_not_syntax_funcs = []
check_source_code_syntax_num = []

for adr, sc in soliaudit_syntax_files:
    lines = sc.splitlines(keepends=True)
    cp = 0
    for i, line in enumerate(lines):
        if "function " in line or "modifier " in line or "constructor " in line:
            cp = 1
            try:
                sampled_func = find_function(lines, i)
                if "block.timestamp" in sampled_func or "now" in sampled_func:
                    time_non_vul_syntax_funcs.append(sampled_func)
                else:
                    time_non_vul_not_syntax_funcs.append(sampled_func)
            except:
                pass
    if cp == 1:
        check_source_code_syntax_num.append(adr)

print("Number of non-vulnerability source codes containing syntax block.timestamp or now:", len(check_source_code_syntax_num))

time_non_vul_syntax_funcs = list(set(time_non_vul_syntax_funcs))
time_non_vul_not_syntax_funcs = list(set(time_non_vul_not_syntax_funcs))

print("Number of non-vulnerability functions containing syntax block.timestamp or now:", len(time_non_vul_syntax_funcs))
print("Number of non-vulnerability functions not containing syntax block.timestamp or now:", len(time_non_vul_not_syntax_funcs))

Number of vulnerability functions: 467
Number of train functions: 373
Number of test functions: 94
4845
Number of vulnerability source codes with 1 block.timestamp or 1 now containing train functions: 1750
Number of vulnerability source codes with 1 block.timestamp or 1 now containing test functions: 429
Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions: 2132
Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions: 534
Number of non-vulnerability source codes containing syntax block.timestamp or now: 808
Number of non-vulnerability functions containing syntax block.timestamp or now: 1194
Number of non-vulnerability functions not containing syntax block.timestamp or now: 8761


In [ ]:
time_non_vul_syntax_funcs = sorted(time_non_vul_syntax_funcs)
time_non_vul_not_syntax_funcs = sorted(time_non_vul_not_syntax_funcs)
soliaudit_syntax_files = sorted(soliaudit_syntax_files)

In [ ]:
best_seed = -1
best_preserve = -1

for seed in range(200):
    random.seed(seed)
    sampled_syntax_funcs = time_non_vul_syntax_funcs.copy()
    random.shuffle(sampled_syntax_funcs)
    split_index = int(0.8 * len(sampled_syntax_funcs))
    time_train_non_vul_syntax_funcs = sampled_syntax_funcs[:split_index]
    time_test_non_vul_syntax_funcs = sampled_syntax_funcs[split_index:]

    # sampled_not_syntax_funcs = random.sample(time_non_vul_not_syntax_funcs, 4000)
    # random.shuffle(sampled_not_syntax_funcs)
    # split_index = int(0.8 * len(sampled_not_syntax_funcs))
    # train_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[:split_index]
    # test_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[split_index:]
    time_train_non_vul_syntax_count = []
    time_test_non_vul_syntax_count = []

    # Count occurrences of train functions in total_reentrancy_code
    for adr, code in soliaudit_syntax_files:
        cp = 0
        for func in time_train_non_vul_syntax_funcs:
            if func in code:
                cp = 1
                break
        for func2 in time_test_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_train_non_vul_syntax_count.append((adr, code))


    # Count occurrences of test functions in total_reentrancy_code
    for adr, code in soliaudit_syntax_files:
        cp = 0
        for func in time_test_non_vul_syntax_funcs:
            if func in code and (adr, code) not in time_train_non_vul_syntax_count:
                cp = 1
                break
        for func2 in time_train_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_test_non_vul_syntax_count.append((adr, code))

    # print("Number of non-vulnerability source codes contain block.timestamp or now in train:", len(time_train_non_vul_syntax_count))
    # print("Number of non-vulnerability source codes contain block.timestamp or now in test:", len(time_test_non_vul_syntax_count))

    if len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count) >= 3.9 and len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count) <= 4.2:
        if len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count) > best_preserve:
            best_preserve = len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count), 2)} with able to retain {len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count)} out of {len(soliaudit_syntax_files)}")
        # break

Seed 67 is compatible ~4.07 with able to retain 482 out of 808
Seed 90 is compatible ~4.0 with able to retain 505 out of 808
Seed 166 is compatible ~3.99 with able to retain 489 out of 808


In [ ]:
print(best_seed)

90


In [ ]:
random.seed(best_seed)
sampled_syntax_funcs = time_non_vul_syntax_funcs.copy()
random.shuffle(sampled_syntax_funcs)
split_index = int(0.8 * len(sampled_syntax_funcs))
time_train_non_vul_syntax_funcs = sampled_syntax_funcs[:split_index]
time_test_non_vul_syntax_funcs = sampled_syntax_funcs[split_index:]

# sampled_not_syntax_funcs = random.sample(time_non_vul_not_syntax_funcs, 4000)
# random.shuffle(sampled_not_syntax_funcs)
# split_index = int(0.8 * len(sampled_not_syntax_funcs))
# train_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[:split_index]
# test_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[split_index:]
time_train_non_vul_syntax_count, time_test_non_vul_syntax_count = [], []
time_train_non_vul_syntax_dict, time_test_non_vul_syntax_dict = {}, {}

# Check occurrences of non-vul train functions in soliaudit_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in soliaudit_syntax_files:
    cp = 0
    for func in time_train_non_vul_syntax_funcs:
        if func in code:
            cp = 1
            break
    for func2 in time_test_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_train_non_vul_syntax_count.append((adr, code))
        if func not in time_train_non_vul_syntax_dict:
            time_train_non_vul_syntax_dict[func] = []
        time_train_non_vul_syntax_dict[func].append((adr, code))


# Check occurrences of non-vul test functions in soliaudit_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in soliaudit_syntax_files:
    cp = 0
    for func in time_test_non_vul_syntax_funcs:
        if func in code and (adr, code) not in time_train_non_vul_syntax_count:
            cp = 1
            break
    for func2 in time_train_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_test_non_vul_syntax_count.append((adr, code))
        if func not in time_test_non_vul_syntax_dict:
            time_test_non_vul_syntax_dict[func] = []
        time_test_non_vul_syntax_dict[func].append((adr, code))

print("Number of non-vulnerability source codes contain block.timestamp or now in train:", len(time_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain block.timestamp or now in test:", len(time_test_non_vul_syntax_count))

Number of non-vulnerability source codes contain block.timestamp or now in train: 404
Number of non-vulnerability source codes contain block.timestamp or now in test: 101


In [ ]:
tmp1, tmp2 = [], []
for func in time_non_vul_not_syntax_funcs:
    train_sc = [k[1] for k in time_train_non_vul_syntax_count]
    test_sc = [k[1] for k in time_test_non_vul_syntax_count]
    cpa = 0
    cpb = 0
    for k in train_sc:
        if func in k:
            cpa = 1
            break
    for k in test_sc:
        if func in k:
            cpb = 1
            break
    if cpa == 1 and cpb == 0:
        tmp1.append(func)
    elif cpa == 0 and cpb == 1:
        tmp2.append(func)

print(len(tmp1))
print(len(tmp2))

# 4567
# 660

4567
660


In [ ]:
train_non_vul_not_syntax_funcs = random.sample(tmp1, int(3200 * 0.8))
test_non_vul_not_syntax_funcs = random.sample(tmp2, int(3200 * 0.2))

In [ ]:
print("Number of vulnerability source codes with 1 block.timestamp or now containing train functions:", len(time_train_vul_count))
print("Number of vulnerability source codes with 1 block.timestamp or now containing test functions:", len(time_test_vul_count))

print("Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions:", len(time_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions:", len(time_test_vul_not_counted))

print("Number of non-vulnerability source codes contain block.timestamp or now in train:", len(time_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain block.timestamp or now in test:", len(time_test_non_vul_syntax_count))

Number of vulnerability source codes with 1 block.timestamp or now containing train functions: 1750
Number of vulnerability source codes with 1 block.timestamp or now containing test functions: 429
Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions: 2132
Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions: 534
Number of non-vulnerability source codes contain block.timestamp or now in train: 404
Number of non-vulnerability source codes contain block.timestamp or now in test: 101


In [ ]:
time_train_vul_total = get_filtered_data(time_train_vul_dict, int(len(time_train_non_vul_syntax_count) * 1/2)) + random.sample(time_train_vul_not_counted, int(len(time_train_non_vul_syntax_count) * 1/2))
time_test_vul_total = get_filtered_data(time_test_vul_dict, int(len(time_test_non_vul_syntax_count) * 1/2)) + random.sample(time_test_vul_not_counted, int(len(time_test_non_vul_syntax_count) * 1/2))
time_train_non_vul_total = time_train_non_vul_syntax_count
time_test_non_vul_total = time_test_non_vul_syntax_count

print("Number of vulnerability source codes in train:", len(time_train_vul_total))
print("Number of vulnerability source codes in test:", len(time_test_vul_total))
print("Number of non-vulnerability source codes in train:", len(time_train_non_vul_total))
print("Number of non-vulnerability source codes in test:", len(time_test_non_vul_total))

Number of vulnerability source codes in train: 404
Number of vulnerability source codes in test: 100
Number of non-vulnerability source codes in train: 404
Number of non-vulnerability source codes in test: 101


In [ ]:
time_train_non_vul_funcs = time_train_non_vul_syntax_funcs + train_non_vul_not_syntax_funcs
time_test_non_vul_funcs = time_test_non_vul_syntax_funcs + test_non_vul_not_syntax_funcs

print("Number of non-vulnerability functions in train:", len(time_train_non_vul_funcs))
print("Number of non-vulnerability functions in test:", len(time_test_non_vul_funcs))

Number of non-vulnerability functions in train: 3515
Number of non-vulnerability functions in test: 879


In [ ]:
print("Number of vulnerability functions in train:", len(time_train_vul_funcs))
print("Number of vulnerability functions in test:", len(time_test_vul_funcs))

Number of vulnerability functions in train: 373
Number of vulnerability functions in test: 94


In [ ]:
cnt = 0
for adr, src in time_train_non_vul_total:
    if "block.timestamp" in src or "now" in src:
        cnt += 1

print(cnt)

404


In [ ]:
cnt = 0
for adr, src in time_train_non_vul_total:
    src = remove_comments(src)
    if "block.timestamp" in src or "now" in src:
        cnt += 1

print(cnt)

404


In [ ]:
cnt = 0
for adr, src in time_train_non_vul_total:
    src = clean_code_formatting(remove_comments(src))
    if "block.timestamp" in src or "now" in src:
        cnt += 1

print(cnt)

404


In [ ]:
for k in time_train_vul_total[:10]:
    print(k[0])

0x6439e643dc316ac4108ebdc6c364a65994b6b1ff
0xfbc23099a8bd0ce4227920dc559fcfe9c7fa3ce3
0xd0a6e6c54dbc68db5db3a091b171a77407ff7ccf
0x6a6581a884c4729586479307e5694bf836617517
0x3d3ce5c8baeeb7e6ded63afcb443272a9703527a
0xf4cae4aec9b4d7682f8cee4d9a273ba063e71366
0xc7f39fe227f862981dd20588641b68ab31342bd4
0x1443616b940aea9fd52add2ebdc6966a4ac5f27d
0xad87e48d553C2308dccaB428537F6d0809593bA4
0x4b667a378c1d9b2134cc4fa02d9cfba2cc2b11d4


In [ ]:
for k in time_test_vul_total[:10]:
    print(k[0])

0x50c5712624b58905c19aee87deca593a2690e3f4
0xc324a2f6b05880503444451b8b27e6f9e63287cb
0xE5b340F7E4b11eAE96D7047e8Bc9322f96093402
0xC1ce17303ef35C128B499eD091F39008B3a57389
0x98008a4168b66261cb112d2d4cd940728d033c2e
0x667088b212ce3d06a1b553a7221e1fd19000d9af
0xaed5a41450b38fc0ea0f6f203a985653fe187d9c
0xdF35D37907D70b896DD602E71B6e22339f0c62D9
0xd652c2c57bb8397a790e89ebc392a1bf4e26450f
0x39aa4006ee5941c0c0e41b924fdafcb2c4c918e8


In [ ]:
for k in time_train_non_vul_total[:10]:
    print(k[0])

0x013302040c3bc03b68fbde41d10b93ccf5f561b2
0x01a7018e6d1fde8a68d12f59b6532fb523b6259d
0x01b5420c9beccf458461af5e0b1a6b072879f630
0x0234ae8f8e5a5aeccff9f633aa8d81aa17677ed0
0x02725836ebf3ecdb1cdf1c7b02fcbbfaa2736af8
0x02ab3549536c140af39ebb1c42f25a8e70b4a10a
0x02cf7ec0178f9cee340e4ec0002cb9aa28a401bd
0x03b7dcf4e018031c39f21a6e99632ccbf72c79a2
0x04113ddfe962271fe5d632ba3a3c3a89b2e92e7e
0x04d01b6145bc9db925ec1e80bc85f936a961210e


In [ ]:
for k in time_test_non_vul_total[:10]:
    print(k[0])

0x0371a82e4a9d0a4312f3ee2ac9c6958512891372
0x056e963f69e5fe8b28591cdc469ad671d245dc0d
0x0821126d74d1b33b5cfdedbea66663868a15800c
0x0af44e2784637218dd1d32a322d44e603a8f0c6a
0x1194f98ccfb7428e77f081da20fd07255dc5dae9
0x18a672e11d637fffadccc99b152f4895da069601
0x1b5d56bfe749e492ae226cf9aa23c1426f828b7b
0x1b5d56bfe749e492ae226cf9aa23c1426f828b7b
0x1dce4fa03639b7f0c38ee5bb6065045edcf9819a
0x1e49ff77c355a3e38d6651ce8404af0e48c5395f


In [ ]:
for f in time_train_non_vul_funcs[:10]:
    print(f.splitlines(keepends=True)[0].strip())
    print("-" * 100)

function releaseVestedTokens(address _adr) public changesToVestingFreezed(_adr) {
----------------------------------------------------------------------------------------------------
function mintWithFreeze(address _to, uint256 _value, uint256 _unfreezeTimestamp, bool _subsequentUnlock) public onlyMinter returns (bool) {
----------------------------------------------------------------------------------------------------
function ReleaseTokenForTeamAdvisersPartners () public onlyOwner {
----------------------------------------------------------------------------------------------------
function recalcBonuses() internal;
----------------------------------------------------------------------------------------------------
function addOwnerFromRecovery(address sender, Proxy identity, address newOwner) public
----------------------------------------------------------------------------------------------------
function burn () external {
--------------------------------------------------------

In [ ]:
for f in time_train_vul_funcs[:10]:
    print(f.splitlines(keepends=True)[0].strip())
    print("-" * 100)

function listAddress( address _user, uint _mincap, uint _maxcap ) public onlyOwner {
----------------------------------------------------------------------------------------------------
modifier ICOTerminated() {
----------------------------------------------------------------------------------------------------
function getAvailableAmount() public constant returns(uint256) {
----------------------------------------------------------------------------------------------------
function ReleaseDate() constant returns (uint) { return Date; }
----------------------------------------------------------------------------------------------------
function _currentDay() internal view returns(uint256) {
----------------------------------------------------------------------------------------------------
function unlock() external {
----------------------------------------------------------------------------------------------------
function reward(address _to, uint256 _value, bool locked, string dat

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

src = "/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable"

cnt = 0
for file in os.listdir(src):
    with open(src + "/" + file, "r") as f:
        data = f.read()
    if "block.timestamp" in data or "now" in data:
        cnt += 1

print(cnt)

cnt = 0
for file in os.listdir(src):
    with open(src + "/" + file, "r") as f:
        data = remove_comments(f.read())
    if "block.timestamp" in data or "now" in data:
        cnt += 1

print(cnt)

403
403


In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

306
306
78
78
2
264
256
64
65
2


In [ ]:
for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset", file))

In [ ]:
for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset", file))

In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

0
0
0
0
0
0
0
0
0
0


In [ ]:
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")

In [ ]:
cnt = 0
for adr, sc in reen_train_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in reen_train_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_train_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
cnt2 = 0
for adr, sc in time_train_non_vul_total:
    if "block.timestamp" in sc or "now" in sc:
        cnt2 += 1
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)
print(cnt2)

26
25
53
34
404


In [ ]:
cnt = 0
for adr, sc in reen_test_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in reen_test_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_test_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_test_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

3
8
17
45


In [ ]:
import pandas as pd

time_train_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_train_vul_funcs]
time_train_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_train_non_vul_funcs]
time_test_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_test_vul_funcs]
time_test_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_test_non_vul_funcs]

# Create dataframes for training set
train_vul_df = pd.DataFrame({'function': time_train_vul_funcs, 'label': 1})
train_non_vul_df = pd.DataFrame({'function': time_train_non_vul_funcs, 'label': 0})
train_df = pd.concat([train_vul_df, train_non_vul_df], ignore_index=True)
train_df = train_df.sample(frac=1).reset_index(drop=True)

# Create dataframes for test set
test_vul_df = pd.DataFrame({'function': time_test_vul_funcs, 'label': 1})
test_non_vul_df = pd.DataFrame({'function': time_test_non_vul_funcs, 'label': 0})
test_df = pd.concat([test_vul_df, test_non_vul_df], ignore_index=True)
test_df = test_df.sample(frac=1).reset_index(drop=True)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/train.csv", "w") as f:
    train_df.to_csv(f, index=False)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/test.csv", "w") as f:
    test_df.to_csv(f, index=False)

In [ ]:
reen_train_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_train_vul_funcs]
reen_train_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_train_non_vul_funcs]
reen_test_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_test_vul_funcs]
reen_test_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_test_non_vul_funcs]

# Create dataframes for training set
train_vul_df = pd.DataFrame({'function': reen_train_vul_funcs, 'label': 1})
train_non_vul_df = pd.DataFrame({'function': reen_train_non_vul_funcs, 'label': 0})
train_df = pd.concat([train_vul_df, train_non_vul_df], ignore_index=True)
train_df = train_df.sample(frac=1).reset_index(drop=True)

# Create dataframes for test set
test_vul_df = pd.DataFrame({'function': reen_test_vul_funcs, 'label': 1})
test_non_vul_df = pd.DataFrame({'function': reen_test_non_vul_funcs, 'label': 0})
test_df = pd.concat([test_vul_df, test_non_vul_df], ignore_index=True)
test_df = test_df.sample(frac=1).reset_index(drop=True)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/train.csv", "w") as f:
    train_df.to_csv(f, index=False)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/test.csv", "w") as f:
    test_df.to_csv(f, index=False)

In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

306
306
73
74
2
404
403
100
93
2


In [ ]:
%%capture
!pip install datasets

In [ ]:
from huggingface_hub import HfApi, Repository
import os
from google.colab import userdata
from datasets import Dataset, DatasetDict
import pandas as pd

# Load your CSV files
train_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/test.csv')

# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create a DatasetDict for train and test
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

repo_name = "Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset"  # Replace with your desired repo ID
dataset.push_to_hub(repo_name)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/408 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset/commit/2faf403dcec48e01fcc39d2aeabf145bbac07c3c', commit_message='Upload dataset', commit_description='', oid='2faf403dcec48e01fcc39d2aeabf145bbac07c3c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset'), pr_revision=None, pr_num=None)

In [ ]:
# !rm -r /content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionDataset/
# !rm -r /content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/

In [ ]:
# Load your CSV files
train_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/test.csv')

# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create a DatasetDict for train and test
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

repo_name = "Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset"  # Replace with your desired repo ID
dataset.push_to_hub(repo_name)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/408 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset/commit/9efc5baba659547274f41aecd6ad81afcd807716', commit_message='Upload dataset', commit_description='', oid='9efc5baba659547274f41aecd6ad81afcd807716', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset'), pr_revision=None, pr_num=None)

In [ ]:
!zip -r /content/SmartContractVulnerabilityDetection.zip /content/drive/MyDrive/SmartContractVulnerabilityDetection

  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/ (stored 0%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/sc_versions.pkl (deflated 65%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ (stored 0%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/ (stored 0%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/ (stored 0%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/ (stored 0%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/21390.sol (deflated 57%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/Bank_attack.sol (deflated 56%)
  adding: con